# GFDataset Generator

Generates exactly **110,000** synthetic game-character face images to replicate the *GFDataset* from the *"Paste You Into Game"* research paper.

**Strategy:**
- 5 LoRA styles × 20 base prompts × 1,100 random seeds = **110,000 images**
- Base model: `black-forest-labs/FLUX.1-schnell` via 🤗 `diffusers`
- Generated at 512 × 512, then PIL-cropped / resized to **256 × 256** (paper spec)
- Saved as JPEG under `GFDataset/<style_name>/img_NNNNNN.jpg`
- Full metadata logged to `GFDataset/metadata.csv`

## Cell 1 – Install Dependencies

Run this once (or skip if already installed).

In [ ]:
# Install / upgrade required packages
import importlib.metadata as _meta, subprocess, sys

def _ver(pkg):
    """Return the currently installed version of *pkg* as a (major, minor) tuple."""
    try:
        return tuple(int(x) for x in _meta.version(pkg).split(".")[:2])
    except _meta.PackageNotFoundError:
        return (0, 0)

# Detect stale pre-installed packages BEFORE upgrading so we know whether
# a kernel restart is needed afterwards.  importlib.metadata reads the
# on-disk dist-info, so it reflects the NEW version immediately after pip
# writes it — unlike `import diffusers; diffusers.__version__` which keeps
# the old module in memory until the kernel is restarted.
_stale = (
    _ver("diffusers")       < (0, 30)   # FluxPipeline added in 0.30
    or _ver("transformers") < (4, 40)   # required for FLUX T5 text-encoders
    or _ver("accelerate")   < (0, 30)   # required for pipeline device placement
    or _ver("sentencepiece") == (0, 0)  # required by T5 tokenizer
)

packages = [
    "diffusers>=0.30.0",
    "transformers>=4.40.0",
    "accelerate>=0.30.0",
    "safetensors",
    "torch",
    "Pillow",
    "tqdm",
    "huggingface_hub>=0.20.0",
    # Required by FLUX.1-schnell T5 text-encoder tokenizer;
    # without these, transformers raises a RuntimeError from its
    # lazy-module loader (_LazyModule.__getattr__) when T5Tokenizer
    # or T5EncoderModel is first accessed.
    "sentencepiece",
    "protobuf",
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade"] + packages
)
print("\u2705 All dependencies installed.")

if _stale:
    # One or more packages are stale in the kernel's memory; running the next
    # cell with old in-memory modules can raise:
    #   diffusers  < 0.30  → ImportError: cannot import name 'FluxPipeline'
    #   transformers < 4.40 → RuntimeError via _LazyModule when T5Tokenizer
    #                          or T5EncoderModel is first accessed
    #   accelerate < 0.30  → errors during pipeline device-placement
    #   sentencepiece missing → RuntimeError: install sentencepiece for T5
    # A kernel restart forces Python to import the freshly installed versions.
    # After the restart, re-run all cells — this cell will exit through the
    # `else` branch because the packages are already up to date.
    print("\ud83d\udd04 Restarting kernel to activate updated packages\u2026")
    import os
    os._exit(0)  # force-exit so the Jupyter kernel manager auto-restarts the process
else:
    print(
        f"\u2139\ufe0f  All packages already at required versions "
        f"(diffusers {_meta.version('diffusers')}, "
        f"transformers {_meta.version('transformers')}, "
        f"accelerate {_meta.version('accelerate')}) \u2014 no restart needed."
    )


## Cell 1b – Hugging Face Authentication

`FLUX.1-schnell` is a **gated model**. Before running this cell you must:

1. Accept the licence at <https://huggingface.co/black-forest-labs/FLUX.1-schnell>
2. Create a **read** token at <https://huggingface.co/settings/tokens>
3. Set the token as an environment variable called `HF_TOKEN`:
   - **RunPod** → *Pod Settings → Environment Variables* → add `HF_TOKEN = hf_…`
   - **Locally** → `export HF_TOKEN=hf_…` in your shell (or `huggingface-cli login`)


In [ ]:
import os
from huggingface_hub import login

# FLUX.1-schnell is a gated model on Hugging Face.
# A valid read-access token stored in the HF_TOKEN environment variable is
# required; without it every call to from_pretrained raises GatedRepoError.
#
# How to set HF_TOKEN on RunPod:
#   Pod Settings → Environment Variables → add  HF_TOKEN = hf_<your_token>
# How to set it locally:
#   export HF_TOKEN=hf_<your_token>   # bash / zsh
#   or run:  huggingface-cli login
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()

if not HF_TOKEN:
    raise EnvironmentError(
        "\n"
        "HF_TOKEN environment variable is not set.\n"
        "FLUX.1-schnell is a gated model — authentication is required:\n"
        "  1. Accept the licence  : https://huggingface.co/black-forest-labs/FLUX.1-schnell\n"
        "  2. Create a read token : https://huggingface.co/settings/tokens\n"
        "  3a. RunPod             : Pod Settings → Environment Variables → HF_TOKEN = hf_…\n"
        "  3b. Locally            : export HF_TOKEN=hf_…  (or run huggingface-cli login)\n"
    )

login(token=HF_TOKEN, add_to_git_credential=False)
print(f"\u2705 Logged in to Hugging Face (token: {HF_TOKEN[:8]}\u2026).")


## Cell 2 – Imports & Configuration

In [ ]:
import os
import csv
import gc
import random
from pathlib import Path

import torch
from diffusers import FluxPipeline
from PIL import Image
from tqdm.auto import tqdm

# ── Reproducibility ──────────────────────────────────────────────────────────
MASTER_SEED = 42          # Only used to initialise the random seed sequence
random.seed(MASTER_SEED)

# ── Generation hyper-parameters ──────────────────────────────────────────────
GENERATE_WIDTH   = 512    # Flux native resolution
GENERATE_HEIGHT  = 512
SAVE_SIZE        = 256    # Paper-specified output resolution
NUM_STEPS        = 4      # Schnell is a distilled 4-step model
GUIDANCE_SCALE   = 0.0    # Schnell uses guidance_scale = 0
IMAGES_PER_COMBO = 1_100  # 5 LoRAs × 20 prompts × 1,100 = 110,000

# ── Output paths ─────────────────────────────────────────────────────────────
OUTPUT_ROOT   = Path("GFDataset")
METADATA_FILE = OUTPUT_ROOT / "metadata.csv"

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Cell 3 – LoRA Dictionary

Each entry specifies:
- `repo`    – Hugging Face Hub repository ID (swap this string to use a different LoRA)
- `weights` – filename of the `.safetensors` adapter inside that repo
- `trigger` – trigger word / phrase prepended to every prompt for that style

In [ ]:
# LoRAs must be compatible with FLUX.1-schnell.
# To swap a LoRA: change the "repo" string and "weights" filename.
loras = {
    "pvc_3d_style": {
        "repo"    : "p1atdev/flux.1-schnell-pvc-style-lora",
        "weights" : "flux.1-schnell-pvc-style-lora.safetensors",
        "trigger" : "pvc style",
    },
    "simple_vector": {
        "repo"    : "furaidosu/flux-lora-simple-vector",
        "weights" : "flux-lora-simple-vector.safetensors",
        "trigger" : "simple vector art",
    },
    "schnell_realism": {
        "repo"    : "Octree/flux-schnell-lora",
        "weights" : "flux-schnell-lora.safetensors",
        "trigger" : "high realism cinematic",
    },
    "anime_style": {
        "repo"    : "aleksa-k/flux-schnell-lora-anime",
        "weights" : "flux-schnell-lora-anime.safetensors",
        "trigger" : "anime style",
    },
    "pixel_art": {
        "repo"    : "goofyai/flux-schnell-lora-pixel-art",
        "weights" : "flux-schnell-lora-pixel-art.safetensors",
        "trigger" : "16-bit pixel art",
    },
}

NUM_LORAS = len(loras)
print(f"Loaded {NUM_LORAS} LoRA styles.")

## Cell 4 – Base Prompts Array (20 diverse character types)

In [ ]:
base_prompts = [
    "A close-up front-facing portrait of a futuristic space marine, battle scars, neutral expression",
    "A close-up front-facing portrait of a fantasy elven rogue, green hood, subtle smile",
    "A close-up front-facing portrait of a cyberpunk hacker, neon reflections on face, serious expression",
    "A close-up front-facing portrait of a medieval knight, visor up, looking determined",
    "A close-up front-facing portrait of a zombie survivalist, dirt on face, angry expression",
    "A close-up front-facing portrait of a generic male NPC villager, plain clothes, neutral expression",
    "A close-up front-facing portrait of a generic female merchant NPC, warm smile",
    "A close-up front-facing portrait of a high fantasy wizard, long beard, glowing eyes",
    "A close-up front-facing portrait of a modern military sniper, face paint, focused gaze",
    "A close-up front-facing portrait of a sci-fi alien diplomat, blue skin, calm expression",
    "A close-up front-facing portrait of an urban street fighter, bruised cheek, shouting",
    "A close-up front-facing portrait of a gothic vampire lord, pale skin, menacing smirk",
    "A close-up front-facing portrait of a post-apocalyptic scavenger, goggles on forehead, looking surprised",
    "A close-up front-facing portrait of a steampunk engineer, brass accessories, winking",
    "A close-up front-facing portrait of a western cowboy outlaw, hat tilted back, stern look",
    "A close-up front-facing portrait of a futuristic android, synthetic skin, glowing blue iris",
    "A close-up front-facing portrait of a pirate captain, eyepatch, laughing",
    "A close-up front-facing portrait of a martial arts monk, shaved head, serene expression",
    "A close-up front-facing portrait of a royal assassin, dark mask pulled down, cold stare",
    "A close-up front-facing portrait of a mad scientist, wild hair, manic grin",
]

NUM_PROMPTS = len(base_prompts)
TOTAL_IMAGES = NUM_LORAS * NUM_PROMPTS * IMAGES_PER_COMBO

print(f"Base prompts    : {NUM_PROMPTS}")
print(f"LoRA styles     : {NUM_LORAS}")
print(f"Images / combo  : {IMAGES_PER_COMBO}")
print(f"Total images    : {NUM_LORAS} × {NUM_PROMPTS} × {IMAGES_PER_COMBO} = {TOTAL_IMAGES:,}")

## Cell 5 – Directory Setup & Metadata Writer Helper

In [ ]:
def setup_directories(lora_names: list[str]) -> None:
    """Create GFDataset/ root and one subfolder per LoRA style."""
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    for name in lora_names:
        (OUTPUT_ROOT / name).mkdir(parents=True, exist_ok=True)
    print(f"Output root     : {OUTPUT_ROOT.resolve()}")


def init_metadata_csv() -> tuple:
    """
    Open (or append to) metadata.csv and return a (DictWriter, file_handle) tuple.
    Writes the header row only if the file is new / empty.
    Returns the writer AND the open file handle so the caller can close it.
    """
    fieldnames = [
        "filename",
        "style_name",
        "base_prompt",
        "full_prompt",
        "seed",
        "resolution_saved",
    ]
    write_header = (not METADATA_FILE.exists()) or (METADATA_FILE.stat().st_size == 0)
    fh = METADATA_FILE.open("a", newline="", encoding="utf-8")
    writer = csv.DictWriter(fh, fieldnames=fieldnames)
    if write_header:
        writer.writeheader()
    return writer, fh


setup_directories(list(loras.keys()))
print("Directory structure ready.")

## Cell 6 – Global Image Counter (resume-safe)

Counts how many images already exist so a re-run can continue from where it left off.

In [ ]:
def count_existing_images() -> int:
    """Return the total number of JPEG images already saved under OUTPUT_ROOT."""
    return sum(1 for _ in OUTPUT_ROOT.rglob("*.jpg"))


images_done_at_start = count_existing_images()
print(f"Images already on disk: {images_done_at_start:,}")
print(f"Images remaining      : {max(0, TOTAL_IMAGES - images_done_at_start):,}")

## Cell 7 – Pipeline Loader Helper

Loads the `FluxPipeline` in `bfloat16` with **CPU model offloading** so that only the active
sub-module lives on GPU at any one time.  This is the safest strategy for a ≥110 k-image run.

In [ ]:
BASE_MODEL_ID = "black-forest-labs/FLUX.1-schnell"


def load_pipeline() -> FluxPipeline:
    """
    Load FLUX.1-schnell fully on GPU in bfloat16.

    The RunPod A6000 has 48 GB VRAM, which comfortably holds FLUX.1-schnell
    (≈15-20 GB in bfloat16) plus VAE decode buffers and LoRA weight overhead.
    Loading all sub-modules directly to the GPU with ``pipe.to("cuda")`` is
    therefore the correct strategy here:

    * Eliminates the CPU↔GPU transfer overhead of ``enable_model_cpu_offload``
      (which was designed for 8 GB consumer cards, not for a 48 GB workstation GPU).
    * Maximises inference throughput for the 110 k-image batch run.
    """
    print(f"Loading base model '{BASE_MODEL_ID}' …")
    pipe = FluxPipeline.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.bfloat16,
        token=os.environ.get("HF_TOKEN"),  # required for gated model
    )
    # Load every sub-module (transformer, VAE, text encoders) to the GPU at
    # once.  With 48 GB of VRAM this is safe and gives the best throughput.
    pipe.to("cuda")
    print("Pipeline loaded to GPU (full model on VRAM — A6000 48 GB).")
    return pipe


def clear_vram() -> None:
    """Release all CUDA tensors and reclaim VRAM between LoRA swaps."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


print("Pipeline helper functions defined.")


## Cell 8 – Image Post-Processing Helper

Generates at 512 × 512 then crops / resizes to 256 × 256 before saving.

In [ ]:
def postprocess_image(pil_image: Image.Image) -> Image.Image:
    """
    Centre-crop and resize a PIL Image to SAVE_SIZE × SAVE_SIZE (256 × 256).

    Strategy:
    1. Centre-crop the image to a square if it is not already square.
    2. Resize to SAVE_SIZE × SAVE_SIZE using high-quality Lanczos resampling.
    """
    w, h = pil_image.size
    # Step 1: centre-crop to square
    min_dim = min(w, h)
    left   = (w - min_dim) // 2
    top    = (h - min_dim) // 2
    right  = left + min_dim
    bottom = top  + min_dim
    pil_image = pil_image.crop((left, top, right, bottom))
    # Step 2: resize to target
    pil_image = pil_image.resize((SAVE_SIZE, SAVE_SIZE), Image.LANCZOS)
    return pil_image


print(f"Post-processing: 512 × 512  →  {SAVE_SIZE} × {SAVE_SIZE} (centre-crop + Lanczos resize)")

## Cell 9 – Main Generation Loop

Outer loop  → LoRA styles (5)  
Middle loop → base prompts (20)  
Inner loop  → random seeds (1,100)

Total: **5 × 20 × 1,100 = 110,000 images**

In [ ]:
# Pre-generate all seeds upfront so the sequence is deterministic
# regardless of interruptions / restarts.
rng = random.Random(MASTER_SEED)
all_seeds = [
    [rng.randint(0, 2**31 - 1) for _ in range(IMAGES_PER_COMBO)]
    for _ in range(NUM_LORAS * NUM_PROMPTS)
]

# Open (or re-open) metadata file
csv_writer, csv_fh = init_metadata_csv()

# Load the base pipeline once
pipe = load_pipeline()

global_img_counter = 0   # running 1-based counter used for filenames
combo_index        = 0   # index into all_seeds

try:
    # ── Outer loop: LoRA styles ───────────────────────────────────────────────
    for style_name, lora_cfg in tqdm(
        loras.items(),
        desc="LoRA styles",
        total=NUM_LORAS,
        unit="style",
    ):
        style_dir = OUTPUT_ROOT / style_name

        # Load LoRA weights for this style
        print(f"\nLoading LoRA '{style_name}' from '{lora_cfg['repo']}' …")
        pipe.load_lora_weights(
            lora_cfg["repo"],
            weight_name=lora_cfg["weights"],
        )

        # ── Middle loop: base prompts ─────────────────────────────────────────
        for prompt_idx, base_prompt in enumerate(
            tqdm(
                base_prompts,
                desc=f"  Prompts [{style_name}]",
                total=NUM_PROMPTS,
                unit="prompt",
                leave=False,
            )
        ):
            # Compose final prompt: trigger word + base description
            full_prompt = f"{lora_cfg['trigger']}, {base_prompt}"
            combo_seeds = all_seeds[combo_index]
            combo_index += 1

            # ── Inner loop: 1,100 seeds per prompt/LoRA combination ───────────
            for seed in tqdm(
                combo_seeds,
                desc=f"    Images [{base_prompt[:40]}…]",
                total=IMAGES_PER_COMBO,
                unit="img",
                leave=False,
            ):
                global_img_counter += 1
                filename = f"img_{global_img_counter:06d}.jpg"
                save_path = style_dir / filename

                # Skip already-generated images (resume support)
                if save_path.exists():
                    continue

                # ── Generate ─────────────────────────────────────────────────
                gen_device = "cuda" if torch.cuda.is_available() else "cpu"
                generator = torch.Generator(device=gen_device).manual_seed(seed)
                result = pipe(
                    prompt            = full_prompt,
                    width             = GENERATE_WIDTH,
                    height            = GENERATE_HEIGHT,
                    num_inference_steps = NUM_STEPS,
                    guidance_scale    = GUIDANCE_SCALE,
                    generator         = generator,
                    num_images_per_prompt = 1,
                )
                pil_image = result.images[0]

                # ── Post-process & save ───────────────────────────────────────
                pil_image = postprocess_image(pil_image)
                pil_image.save(save_path, format="JPEG", quality=95)

                # ── Log metadata ──────────────────────────────────────────────
                csv_writer.writerow({
                    "filename"        : str(save_path.relative_to(OUTPUT_ROOT)),
                    "style_name"      : style_name,
                    "base_prompt"     : base_prompt,
                    "full_prompt"     : full_prompt,
                    "seed"            : seed,
                    "resolution_saved": f"{SAVE_SIZE}x{SAVE_SIZE}",
                })
                csv_fh.flush()   # flush after every write for crash safety

        # ── Unload LoRA & clear VRAM before next style ────────────────────────
        print(f"Unloading LoRA '{style_name}' and clearing VRAM …")
        pipe.unload_lora_weights()
        clear_vram()

finally:
    # Always close the CSV file even if the run is interrupted
    csv_fh.close()
    print("\nCSV file closed.")

total_on_disk = count_existing_images()
print(f"\n✅ Generation complete.")
print(f"   Images on disk : {total_on_disk:,} / {TOTAL_IMAGES:,}")
print(f"   Metadata CSV   : {METADATA_FILE.resolve()}")

## Cell 10 – Sanity Check

Verify the final count and display a few sample images.

In [ ]:
import math
from IPython.display import display

# Count images per style
print("Images per style:")
style_counts = {}
for style_name in loras:
    count = sum(1 for _ in (OUTPUT_ROOT / style_name).glob("*.jpg"))
    style_counts[style_name] = count
    expected = NUM_PROMPTS * IMAGES_PER_COMBO
    status = "✅" if count == expected else "⚠️ "
    print(f"  {status} {style_name:<20} {count:>7,} / {expected:,}")

grand_total = sum(style_counts.values())
print(f"\n  Grand total: {grand_total:,} / {TOTAL_IMAGES:,}")

# Display a 5-image sample strip (one per style)
samples = []
for style_name in loras:
    jpgs = sorted((OUTPUT_ROOT / style_name).glob("*.jpg"))
    if jpgs:
        samples.append(Image.open(jpgs[0]))

if samples:
    strip_w = SAVE_SIZE * len(samples)
    strip   = Image.new("RGB", (strip_w, SAVE_SIZE))
    for i, img in enumerate(samples):
        strip.paste(img, (i * SAVE_SIZE, 0))
    print("\nSample images (one per style):")
    display(strip)

## Cell 11 – Metadata Preview

In [ ]:
# Quick preview of the first few rows of metadata.csv
with METADATA_FILE.open("r", encoding="utf-8") as fh:
    rows = fh.readlines()

print(f"metadata.csv  —  {len(rows) - 1:,} data rows  (+ 1 header)")
print()
# Pretty-print the header + first 5 rows
for line in rows[:6]:
    print(line.rstrip())